# 00 — Setup

Creates the Unity Catalog schema (idempotent), ensures landing volume parent exists, and grants `CREATE MODEL` so MLflow can register models in UC.

Runs on the Shared all-purpose cluster.

### Step 1 — Configure catalog and schema

Read the job widgets for `catalog` and `schema`, switch the Spark session to that catalog, and create the Unity Catalog schema if it does not already exist. This is idempotent so re-runs are safe.

In [ ]:
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "ml")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}` COMMENT 'Early claim severity ML demo'")
print(f"Schema ready: {catalog}.{schema}")

### Step 2 — Grant demo privileges

Apply best-effort `GRANT`s so account users can use the catalog/schema, create tables and volumes, register models, and read/write demo tables. Failures are skipped when the caller lacks admin rights — common in shared workspaces.

In [ ]:
# Grant model + table privileges for demo users (idempotent best-effort).
grants = [
    f"GRANT USE_CATALOG ON CATALOG `{catalog}` TO `account users`",
    f"GRANT USE_SCHEMA ON SCHEMA `{catalog}`.`{schema}` TO `account users`",
    f"GRANT CREATE TABLE ON SCHEMA `{catalog}`.`{schema}` TO `account users`",
    f"GRANT CREATE MODEL ON SCHEMA `{catalog}`.`{schema}` TO `account users`",
    f"GRANT CREATE VOLUME ON SCHEMA `{catalog}`.`{schema}` TO `account users`",
    f"GRANT SELECT ON SCHEMA `{catalog}`.`{schema}` TO `account users`",
    f"GRANT MODIFY ON SCHEMA `{catalog}`.`{schema}` TO `account users`",
]

for stmt in grants:
    try:
        spark.sql(stmt)
        print(f"OK: {stmt}")
    except Exception as exc:  # noqa: BLE001 — demo notebook; permissions vary by caller
        print(f"SKIP ({type(exc).__name__}): {stmt}")

print("Setup complete.")